# Notebook 03 — Full FD vs Neural-Network Comparison

**Goal:** Rigorous comparison of the corrected FD HJB viscosity solver against neural network methods  
across **n = 1, 5, 10, 20** assets, evaluated by **Monte Carlo simulation** (500 GBM paths each).

### Methods compared
| ID | Family | Description |
|---|---|---|
| `fd_1d_proxy` | FD | Corrected viscosity HJB on 1-D equal-risk aggregate (Dai et al. 2019) |
| `browne_1d` | Analytical | Browne (1995) goal-reaching closed form on 1-D aggregate |
| `equal_weight` | Baseline | 1/n equal-weight static |
| `nn_mlp_small` | NN | 2-hidden-layer MLP (32×32), PyTorch, BSDE training |
| `nn_mlp_deep` | NN | 3-hidden-layer MLP (64×64×32), PyTorch |

### Metrics reported
- Target-hit probability P(W_T ≥ 1.10)
- Mean / median terminal wealth
- Mean shortfall E[max(1.10 − W_T, 0)]
- Sharpe ratio (annualised, excess over r=3%)
- Policy π(w) vs wealth at τ=1 and τ=0.5
- Terminal wealth histograms
- Scalability: target-hit rate vs n_assets
- Compute cost: solve/train time vs n_assets

In [1]:
# ── Cell 1: Path bootstrap (same pattern as notebooks 01 & 02) ──────────
import sys
from pathlib import Path
import math

ROOT = Path.cwd().resolve()
while not (ROOT / 'comparisons').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('ROOT =', ROOT)
print('Python:', sys.version.split()[0])

ROOT = /Users/ahmedalqubaisi/Desktop/KU/PhD/Code/Claude Code
Python: 3.9.10


In [2]:
# ── Cell 2: Imports ─────────────────────────────────────────────────────
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

matplotlib.rcParams.update({
    'figure.dpi': 120, 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
})

import comparisons  # triggers __init__.py sys.path bootstrap
from fd_core import (
    asymp_goalreach, fd_solve, fd_solve_nd,
    goal_utility, make_fd_policy, make_fd_policy_nd,
    pi_browne_nd, browne_V_nd,
)
from real_data_loader import agg_1d, load_portfolio
from comparisons.core.evaluation import browne_policy_1d

# PyTorch (optional - falls back to numpy NN if absent)
HAS_TORCH = False
try:
    import torch
    from comparisons.core.torch_nn_models import (
        TORCH_ARCHITECTURES,
        policy_weights as torch_policy_weights,
        train_torch_policy_net,
    )
    HAS_TORCH = True
    if torch.cuda.is_available():
        DEVICE = torch.device('cuda')
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        DEVICE = torch.device('mps')   # Apple Silicon GPU
    else:
        DEVICE = torch.device('cpu')
    print(f'\u2713 PyTorch {torch.__version__}  device={DEVICE}')
except ImportError:
    from comparisons.core.nn_models import (
        ARCHITECTURES as TORCH_ARCHITECTURES,
        policy_weights as torch_policy_weights,
        train_numpy_policy_net as train_torch_policy_net,
    )
    print('\u26a0 PyTorch not found \u2014 using numpy NN fallback')

print('All imports OK')

✓ PyTorch 2.8.0  device=mps
All imports OK


In [3]:
# ── Cell 3: Experiment parameters ───────────────────────────────────────

N_ASSETS_LIST  = [1, 5, 10, 20]   # asset universe sizes
SEEDS          = [1, 2, 3]         # random seeds
INITIAL_WEALTH = 1.0
TARGET_MULT    = 1.10              # goal = 1.10 × W0
N_MC           = 500               # MC paths per evaluation (set to 200 for speed)

# Per-asset bounds and aggregate leverage caps
D, U           = -5.0, 3.0
MAX_LONG       = 3.0
MAX_SHORT      = 5.0

# FD grid resolution
FD_NW, FD_NT   = 120, 80
FD_WMAX        = 2.5

# NN training settings
NN_ITERS       = 50   # reduce to 20 for a quicker run
NN_PATHS       = 512
NN_STEPS       = 32
NN_LR          = 3e-3
NN_ARCHS       = ['nn_mlp_small', 'nn_mlp_deep']

GOAL = INITIAL_WEALTH * TARGET_MULT

# Method display names / colours
# fd_1d  = exact Dai et al. (2019) 1-D solver  (n=1 only)
# fd_nd  = proper multi-asset extension         (n>1, same grid, QP per node)
METHOD_META = {
    'fd_1d':        dict(label='FD HJB (1D, exact)',       color='steelblue',   ls='-',  lw=2.5, marker='o'),
    'fd_nd':        dict(label='FD HJB (nD, proper)',      color='dodgerblue',  ls='-',  lw=2.5, marker='o'),
    'browne_1d':    dict(label='Browne (1D)',               color='darkorange',  ls='--', lw=2,   marker='s'),
    'browne_nd':    dict(label='Browne (nD)',               color='chocolate',   ls='--', lw=2,   marker='s'),
    'equal_weight': dict(label='Equal-weight',             color='gray',        ls=':',  lw=1.5, marker='^'),
    'nn_mlp_small': dict(label='MLP-small (32\u00d732)',   color='crimson',     ls='-',  lw=2,   marker='D'),
    'nn_mlp_deep':  dict(label='MLP-deep (64\u00d764\u00d732)', color='purple', ls='--', lw=2,  marker='v'),
}

print(f'Config: n_assets={N_ASSETS_LIST}  seeds={SEEDS}  N_MC={N_MC}')
print(f'Goal multiplier: {TARGET_MULT}   D/U: [{D},{U}]   MaxLong/Short: {MAX_LONG}/{MAX_SHORT}')


Config: n_assets=[1, 5, 10, 20]  seeds=[1, 2, 3]  N_MC=500
Goal multiplier: 1.1   D/U: [-5.0,3.0]   MaxLong/Short: 3.0/5.0


In [4]:
# ── Cell 4: Helper functions ─────────────────────────────────────────────

def clip_leverage(weights, d=D, u=U, max_long=MAX_LONG, max_short=MAX_SHORT):
    """Per-asset clip then aggregate long/short cap."""
    weights = np.clip(weights, d, u)
    squeeze = weights.ndim == 1
    if squeeze:
        weights = weights[None, :]
    lp = np.maximum(weights, 0).sum(1, keepdims=True).clip(min=1e-12)
    sp = np.maximum(-weights, 0).sum(1, keepdims=True).clip(min=1e-12)
    weights = np.where(weights >= 0,
                       weights * np.minimum(max_long / lp, 1.0),
                       weights * np.minimum(max_short / sp, 1.0))
    return weights.squeeze(0) if squeeze else weights


# ── Vectorised MC runners ─────────────────────────────────────────────────────
#
# Old approach:  Python loop over N_MC paths  →  N_MC × 252 Python iters
# New approach:  pre-generate all noise, numpy array ops at each timestep
#                → only 252 Python iters regardless of N_MC
#
# Speedup:  ~N_MC × (time_per_iter / numpy_vectorised_step) ≈ 100–500×
#
# Policy function signatures expected by the vectorised runners:
#   1-D:  policy(w_norm_arr: (N,), tau) -> (N,)   scalar output also accepted
#   n-D:  policy(W_arr: (N,), goal)    -> (N, n)  or (n,) constant broadcast
#
# Both make_fd_policy and make_fd_policy_nd were updated to support array input.
# Browne ND uses the vectorised _browne_nd_vec helper below.

def _browne_nd_vec(W_arr, tau, eta_nd, omega_mat, goal=GOAL, d=D, u=U,
                   max_long=MAX_LONG, max_short=MAX_SHORT):
    """Fully vectorised multi-asset Browne policy.
    W_arr : (N,) current wealth  (NOT normalised - uses goal internally)
    Returns Pi : (N, n)
    """
    omega_inv_eta = np.linalg.solve(omega_mat, eta_nd)
    theta2 = max(float(eta_nd @ omega_inv_eta), 1e-12)
    tau_safe = max(tau, 1e-10)
    log_ratio = np.log(np.maximum(W_arr / goal, 1e-10))   # (N,)
    denom = 1.0 + log_ratio / (theta2 * tau_safe)          # (N,)
    pi_1d = np.where(np.abs(denom) > 1e-10, 1.0 / denom, 0.0)   # (N,)
    Pi = pi_1d[:, None] * omega_inv_eta[None, :]            # (N, n)
    # batch leverage clip
    Pi = np.clip(Pi, d, u)
    lp = np.maximum(Pi, 0).sum(axis=1, keepdims=True).clip(min=1e-12)
    sp = np.maximum(-Pi, 0).sum(axis=1, keepdims=True).clip(min=1e-12)
    Pi = np.where(Pi >= 0,
                  Pi * np.minimum(max_long  / lp, 1.0),
                  Pi * np.minimum(max_short / sp, 1.0))
    return Pi


def run_mc_1d_vec(policy_fn, mu, r, sigma, n_mc, seed, days=252):
    """Vectorised 1-D GBM MC - all paths updated simultaneously at each step.
    policy_fn(w_norm_arr, tau) should return array of shape (n_mc,).
    Falls back gracefully if policy returns a scalar (constant policy).
    """
    rng = np.random.default_rng(seed)
    dt = 1.0 / days
    Z  = rng.standard_normal((n_mc, days))                  # all noise upfront
    # Pre-compute log returns for each path × day
    log_ret = (mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z  # (n_mc, days)
    r_daily = r * dt

    W = np.full(n_mc, float(INITIAL_WEALTH))
    for t in range(days):
        tau    = max(1.0 - t*dt, dt)
        w_norm = W / GOAL                                    # (n_mc,)
        try:
            pi_raw = policy_fn(w_norm, tau)
            pi = np.clip(np.broadcast_to(
                     np.asarray(pi_raw, float), (n_mc,)).copy(), D, U)
        except (TypeError, ValueError):
            # Scalar-only policy: call per-path (slower but always correct)
            pi = np.clip([float(policy_fn(float(w), tau)) for w in w_norm], D, U)
        # Match simulate_one_year_1asset discretisation exactly
        gross      = np.exp(log_ret[:, t]) - 1              # (n_mc,)
        ret_excess = pi*gross + (1 - pi)*(np.exp(r_daily) - 1)
        W = np.maximum(W*(1 + ret_excess), 1e-6)
    return W


def run_mc_nd_vec(policy_fn, mkt, n_mc, seed, days=252, target_vol=0.25):
    """Vectorised n-D GBM MC - all paths updated simultaneously at each step.
    policy_fn(W_arr, goal) should return (n_mc, n) or (n,) constant policy.
    Matches simulate_one_year_5asset discretisation exactly.
    """
    rng = np.random.default_rng(seed)
    n   = mkt.n
    dt  = 1.0 / days
    L   = np.linalg.cholesky(mkt.omega)                     # (n, n)
    Z   = rng.standard_normal((n_mc, days, n))               # (n_mc, days, n)
    # log returns: (n_mc, days, n)
    log_rets = ((mkt.mu_ann - 0.5*np.diag(mkt.omega))*dt
                + (Z @ L.T)*np.sqrt(dt))
    r_daily  = mkt.r * dt

    W = np.full(n_mc, float(INITIAL_WEALTH))
    for t in range(days):
        tau    = max(1.0 - t*dt, dt)
        pi_raw = policy_fn(W, GOAL, tau=tau)                 # (n_mc,n) or (n,)
        Pi = np.asarray(pi_raw, float)
        if Pi.ndim == 1 and Pi.shape[0] == n:
            Pi = np.tile(Pi, (n_mc, 1))                      # broadcast constant
        # variance scaling - match simulate_one_year_5asset
        port_var = np.einsum('bi,ij,bj->b', Pi, mkt.omega, Pi)  # (n_mc,)
        scale = np.where(port_var > 1e-8,
                         np.minimum(target_vol / np.sqrt(np.maximum(port_var, 1e-8)), 2.0),
                         1.0)
        Pi = np.clip(Pi * scale[:, None], D, U)              # (n_mc, n)
        gross  = np.exp(log_rets[:, t, :]) - 1               # (n_mc, n)
        excess = (Pi * (gross - r_daily)).sum(axis=1)         # (n_mc,)
        W = np.maximum(W*(1 + r_daily + excess), 1e-6)
    return W


# Keep old scalar runners as thin wrappers (backward compat for other notebooks)
def run_mc_1d(policy_fn, mu, r, sigma, n_mc, seed):
    return run_mc_1d_vec(policy_fn, mu, r, sigma, n_mc, seed)

def run_mc_nd(policy_fn, mkt, n_mc, seed):
    return run_mc_nd_vec(policy_fn, mkt, n_mc, seed)


def mc_summary(terminal_wealth, goal=GOAL):
    """Compute scalar summary metrics from MC terminal wealth array."""
    hit = terminal_wealth >= goal
    shortfall = np.maximum(goal - terminal_wealth, 0.0)
    return dict(
        target_hit_prob        = float(hit.mean()),
        mean_terminal_wealth   = float(terminal_wealth.mean()),
        median_terminal_wealth = float(np.median(terminal_wealth)),
        shortfall_mean         = float(shortfall.mean()),
        pct5_wealth            = float(np.percentile(terminal_wealth, 5)),
        pct95_wealth           = float(np.percentile(terminal_wealth, 95)),
    )


def solve_fd_1d(mu_1d, sig_1d, r):
    """1-D Dai et al. (2019) solver - exact for n=1."""
    t0 = time.perf_counter()
    w_grid, _, pi_grid = fd_solve(
        mu=mu_1d, r=r, sigma=sig_1d, T=1.0,
        A=FD_WMAX, Nw=FD_NW, Nt=FD_NT, d=D, u=U,
        utility_fn=goal_utility,
        asymptotic_fn=lambda w, tau: asymp_goalreach(w, tau, sig_1d, D, U),
        UB=0.0, UA=1.0,
    )
    dt = time.perf_counter() - t0
    policy = make_fd_policy(w_grid, pi_grid, d=D, u=U)
    Pi_grid = pi_grid[:, None]   # shape (Nw+1, 1) - consistent with nd output
    return policy, w_grid, Pi_grid, dt


def solve_fd_nd(mkt):
    """Proper multi-asset HJB solver (Dai et al. extended to n assets).
    State: scalar wealth w.  Policy: pi*(w) in R^n via QP at each node.
    Complexity: O(Nw x Nt x n^2).
    """
    n = mkt.n
    w_eq    = np.ones(n) / n
    sig_agg = float(np.sqrt(w_eq @ mkt.omega @ w_eq))
    t0 = time.perf_counter()
    w_grid, _, Pi_grid = fd_solve_nd(
        mu_vec=mkt.mu_ann, r=mkt.r, omega_mat=mkt.omega,
        T=1.0, A=FD_WMAX, Nw=FD_NW, Nt=FD_NT, d=D, u=U,
        utility_fn=goal_utility,
        asymptotic_fn=lambda w, tau: asymp_goalreach(w, tau, sig_agg, D, U),
        max_long=MAX_LONG, max_short=MAX_SHORT,
        UB=0.0, UA=1.0,
    )
    dt = time.perf_counter() - t0
    policy = make_fd_policy_nd(w_grid, Pi_grid, d=D, u=U)
    return policy, w_grid, Pi_grid, dt


print('Helper functions defined.')

Helper functions defined.


In [5]:
# ── Cell 5: Load market data and inspect ────────────────────────────────

mkts = {n: load_portfolio(n) for n in N_ASSETS_LIST}

print('Market data loaded:')
for n, mkt in mkts.items():
    mu_1d, sig_1d, _ = agg_1d(mkt)
    print(f'  n={n:2d}  1D-agg: mu={mu_1d*100:.1f}%  sigma={sig_1d*100:.1f}%  '
          f'r={mkt.r*100:.1f}%  Sharpe≈{(mu_1d-mkt.r)/sig_1d:.2f}')

Market data loaded:
  n= 1  1D-agg: mu=12.6%  sigma=17.3%  r=3.0%  Sharpe≈0.56
  n= 5  1D-agg: mu=9.6%  sigma=11.4%  r=3.0%  Sharpe≈0.58
  n=10  1D-agg: mu=8.5%  sigma=11.4%  r=3.0%  Sharpe≈0.49
  n=20  1D-agg: mu=6.0%  sigma=10.2%  r=3.0%  Sharpe≈0.30


In [6]:
# ── Cell 6: Solve FD HJB for each n ─────────────────────────────────────
# n=1  : exact 1-D Dai et al. (2019)          - fd_solve,    O(Nw·Nt)
# n>1  : proper multi-asset extension          - fd_solve_nd, O(Nw·Nt·n²)
#
# Both share the same (Nw, Nt) grid; fd_solve_nd adds a QP per node.
#
# Browne analytical reference:
#   n=1  : exact 1-D Browne (1995)  - browne_policy_1d
#   n>1  : multi-asset Browne       - pi_browne_nd
#          Direction: Omega^{-1}eta (tangency portfolio)
#          Magnitude: 1-D Browne scalar with sigma_eff=theta=sqrt(eta^T Omega^{-1} eta)

fd_results      = {}   # n -> {policy, w_grid, Pi_grid, solve_sec, method_name}
browne_policies = {}   # n -> callable  policy(w_norm, tau) or policy(W, goal)
browne_is_nd    = {}   # n -> bool  (True if multi-asset Browne, False if 1-D aggregate)

for n in N_ASSETS_LIST:
    mkt = mkts[n]
    mu_1d, sig_1d, _ = agg_1d(mkt)
    r = mkt.r

    # ── FD solver ────────────────────────────────────────────────────────
    if n == 1:
        policy, w_grid, Pi_grid, dt = solve_fd_1d(mu_1d, sig_1d, r)
        method_name = 'fd_1d'
    else:
        policy, w_grid, Pi_grid, dt = solve_fd_nd(mkt)
        method_name = 'fd_nd'

    fd_results[n] = dict(
        policy=policy, w_grid=w_grid, Pi_grid=Pi_grid,
        solve_sec=dt, method_name=method_name,
    )

    # ── Browne analytical reference ──────────────────────────────────────
    if n == 1:
        browne_policies[n] = browne_policy_1d(mu_1d, r, sig_1d, d=D, u=U)
        browne_is_nd[n]    = False
    else:
        eta_nd = mkt.mu_ann - mkt.r
        # Capture mkt by value so the closure is correct inside the loop
        def _make_browne_nd(eta, r_val, omega):
            def _pol(W, goal, tau=1.0, _eta=eta, _r=r_val, _omega=omega):
                # Vectorised: W is (N_MC,) array, returns (N_MC, n)
                return _browne_nd_vec(W, tau, _eta, _omega, goal=goal, d=D, u=U)
            return _pol
        browne_policies[n] = _make_browne_nd(eta_nd, r, mkt.omega)
        browne_is_nd[n]    = True

    print(f'n={n:2d}  [{method_name}]  FD {dt:.2f}s  '
          f'browne={"nd" if browne_is_nd[n] else "1d-agg"}  '
          f'theta={np.sqrt(max(float((mkt.mu_ann-r) @ np.linalg.solve(mkt.omega, mkt.mu_ann-r)), 0)):.3f}')

print('\nAll FD systems solved.')

n= 1  [fd_1d]  FD 0.41s  browne=1d-agg  theta=0.556


n= 5  [fd_nd]  FD 0.89s  browne=nd  theta=0.795


n=10  [fd_nd]  FD 0.94s  browne=nd  theta=1.145


n=20  [fd_nd]  FD 1.26s  browne=nd  theta=3.002

All FD systems solved.


In [7]:
# ── Cell 7: Plot FD policy curves for each n ─────────────────────────────
# n=1 : single policy π(w) for several τ values (exact 1-D)
# n>1 : per-asset weights at τ=1 from the multi-asset FD grid

taus   = [1.0, 0.5, 0.25, 0.1]
tau_colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(taus)))
asset_colors = plt.cm.tab10(np.linspace(0, 0.9, 10))
w_plot = np.linspace(0.2, 1.5, 300)

fig, axes = plt.subplots(1, len(N_ASSETS_LIST), figsize=(15, 4), sharey=False)

for ax, n in zip(axes, N_ASSETS_LIST):
    w_grid  = fd_results[n]['w_grid']
    Pi_grid = fd_results[n]['Pi_grid']   # (Nw+1, k) where k=n
    policy  = fd_results[n]['policy']
    mname   = fd_results[n]['method_name']
    k       = Pi_grid.shape[1]

    if n == 1:
        # Single asset: plot multiple tau slices
        for tau, c in zip(taus, tau_colors):
            pi_fd = np.array([float(policy(w, tau)) for w in w_plot])
            ax.plot(w_plot, pi_fd, color=c, lw=1.8, label=f'FD \u03c4={tau}')
        pi_br = np.array([browne_policies[n](w, 1.0) for w in w_plot])
        ax.plot(w_plot, pi_br, 'r--', lw=1.5, label='Browne \u03c4=1')
        ax.set_title(f'n=1  [fd_1d — exact]', fontsize=10)
    else:
        # Multi-asset: plot per-asset weight at tau=1 from Pi_grid
        tickers = mkts[n].tickers
        show = min(k, 8)
        for i in range(show):
            pi_i = np.interp(w_plot, w_grid, Pi_grid[:, i])
            ax.plot(w_plot, pi_i, color=asset_colors[i % 10], lw=1.4,
                    label=tickers[i], alpha=0.85)
        # Total allocation
        pi_total = sum(np.interp(w_plot, w_grid, Pi_grid[:, i]) for i in range(k))
        ax.plot(w_plot, pi_total, 'k-', lw=2.0, label='Total \u03a3\u03c0', zorder=5)
        if k > 8:
            ax.set_title(f'n={n}  [fd_nd] — top 8 assets', fontsize=10)
        else:
            ax.set_title(f'n={n}  [fd_nd]', fontsize=10)

    ax.axvline(1.0, color='k', lw=0.8, ls=':', alpha=0.6)
    ax.set_xlabel('w = W / goal')
    ax.set_ylim(-6, 4)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, ncol=2)

axes[0].set_ylabel('Portfolio weight \u03c0')
fig.suptitle('FD HJB Policy  [n=1: exact 1-D Dai et al. | n>1: proper multi-asset QP]',
             y=1.02, fontsize=11)
plt.tight_layout()
plt.show()


In [8]:
# ── Cell 8: Train neural networks ────────────────────────────────────────
# For n=1: NN learns pi(W, goal) -> scalar
# For n>1: NN learns pi(W, goal) -> (n,) vector

nn_policies = {}   # (n, arch, seed) -> infer_fn
nn_train_sec = {}  # (n, arch, seed) -> seconds
nn_params = {}     # (n, arch) -> param count

for n in N_ASSETS_LIST:
    mkt = mkts[n]
    for arch in NN_ARCHS:
        for seed in SEEDS:
            key = (n, arch, seed)
            print(f'Training {arch} n={n} seed={seed}...', end=' ', flush=True)
            t0 = time.perf_counter()
            try:
                if HAS_TORCH and arch in TORCH_ARCHITECTURES:
                    net, meta = train_torch_policy_net(
                        mu_vec=mkt.mu_ann, omega_mat=mkt.omega, r=mkt.r,
                        architecture_name=arch,
                        w0=INITIAL_WEALTH, goal_mult=TARGET_MULT,
                        n_paths=NN_PATHS, n_iters=NN_ITERS,
                        n_steps=NN_STEPS, lr=NN_LR,
                        d=D, u=U,
                        max_long_leverage=MAX_LONG,
                        max_short_leverage=MAX_SHORT,
                        seed=seed,
                    )
                    def _infer(W, g, _net=net):
                        raw = np.asarray(torch_policy_weights(_net, W, g), dtype=float)
                        return clip_leverage(raw)
                    infer_fn = _infer
                else:
                    from comparisons.core.nn_models import (
                        policy_weights as np_pw,
                        train_numpy_policy_net as np_train,
                    )
                    net, meta = np_train(
                        mu_vec=mkt.mu_ann, omega_mat=mkt.omega, r=mkt.r,
                        architecture_name=arch,
                        w0=INITIAL_WEALTH, goal_mult=TARGET_MULT,
                        n_paths=NN_PATHS, n_iters=NN_ITERS,
                        population_size=24, elite_frac=0.25, n_steps=NN_STEPS,
                        d=D, u=U,
                        max_long_leverage=MAX_LONG,
                        max_short_leverage=MAX_SHORT,
                        seed=seed,
                    )
                    def _infer(W, g, _net=net):
                        raw = np.asarray(np_pw(_net, W, g), dtype=float)
                        return clip_leverage(raw)
                    infer_fn = _infer

                train_sec = time.perf_counter() - t0
                nn_policies[key]   = infer_fn
                nn_train_sec[key]  = train_sec
                nn_params[(n,arch)] = int(meta.get('param_size', 0))
                print(f'{train_sec:.1f}s  params={nn_params[(n,arch)]}')
            except Exception as exc:
                print(f'FAILED: {exc}')
                nn_policies[key] = None

print('\nAll NN training complete.')

Training nn_mlp_small n=1 seed=1... 

3.1s  params=1185
Training nn_mlp_small n=1 seed=2... 

2.4s  params=1185
Training nn_mlp_small n=1 seed=3... 

2.3s  params=1185
Training nn_mlp_deep n=1 seed=1... 

2.7s  params=6465
Training nn_mlp_deep n=1 seed=2... 

...

2.4s  params=1812
Training nn_mlp_small n=20 seed=3... 

2.5s  params=1812
Training nn_mlp_deep n=20 seed=1... 

2.9s  params=7092
Training nn_mlp_deep n=20 seed=2... 

2.7s  params=7092
Training nn_mlp_deep n=20 seed=3... 

2.8s  params=7092

All NN training complete.


In [9]:
# ── Cell 9: Monte Carlo evaluation (vectorised) ──────────────────────────
#
# run_mc_1d_vec / run_mc_nd_vec loop over 252 timesteps only - all N_MC
# paths are updated simultaneously via numpy array ops.  This is ~N_MC×
# faster than the old per-path Python loop (~500× speedup expected).
#
# FD policy:   policy(w_norm_arr, tau)  ->  (N_MC,) [1D] or (N_MC,n) [nD]
# Browne nD:   _browne_nd_vec(W, tau, ...) ->  (N_MC, n)  [fully vectorised]
# NN policy:   infer_fn(W, goal)         ->  (N_MC,) or (N_MC,n)  [batched]

all_rows = []
t_mc_start = time.perf_counter()

for n in N_ASSETS_LIST:
    mkt      = mkts[n]
    mu_1d, sig_1d, _ = agg_1d(mkt)
    r        = mkt.r
    fd_pol   = fd_results[n]['policy']
    fd_mname = fd_results[n]['method_name']   # 'fd_1d' or 'fd_nd'
    br_pol   = browne_policies[n]
    br_is_nd = browne_is_nd[n]

    for seed in SEEDS:
        base = dict(n_assets=n, seed=seed)

        # ── FD benchmark ──────────────────────────────────────────────────
        if n == 1:
            # policy(w_norm_arr, tau) -> (N_MC,) - make_fd_policy now vectorised
            tw_fd = run_mc_1d_vec(
                policy_fn=fd_pol,
                mu=float(mkt.mu_ann[0]), r=r, sigma=float(mkt.sigma[0]),
                n_mc=N_MC, seed=seed,
            )
        else:
            # policy(w_norm_arr, tau) -> (N_MC, n) - make_fd_policy_nd now vectorised
            def _fd_nd_pol(W, goal, tau=1.0, _fn=fd_pol, _g=GOAL):
                return _fn(W / _g, tau)
            tw_fd = run_mc_nd_vec(
                policy_fn=_fd_nd_pol,
                mkt=mkt, n_mc=N_MC, seed=seed,
            )
        all_rows.append({**base, 'method': fd_mname,
                         'solve_sec': fd_results[n]['solve_sec'], 'train_sec': 0.0,
                         **mc_summary(tw_fd), 'terminal_wealth': tw_fd})

        # ── Browne analytical reference ────────────────────────────────────
        if not br_is_nd:
            # n=1: browne_policy_1d returns scalar policy(w_norm, tau)
            tw_br = run_mc_1d_vec(
                policy_fn=br_pol,
                mu=mu_1d, r=r, sigma=sig_1d,
                n_mc=N_MC, seed=seed,
            )
        else:
            # n>1: _browne_nd_vec already batched: policy(W, goal) -> (N_MC, n)
            tw_br = run_mc_nd_vec(
                policy_fn=br_pol,
                mkt=mkt, n_mc=N_MC, seed=seed,
            )
        br_label = 'browne_nd' if br_is_nd else 'browne_1d'
        all_rows.append({**base, 'method': br_label,
                         'solve_sec': 0.0, 'train_sec': 0.0,
                         **mc_summary(tw_br), 'terminal_wealth': tw_br})

        # ── Equal-weight baseline ──────────────────────────────────────────
        ew = np.ones(n) / n
        if n == 1:
            tw_ew = run_mc_1d_vec(
                policy_fn=lambda w_norm, tau: float(ew[0]),
                mu=float(mkt.mu_ann[0]), r=r, sigma=float(mkt.sigma[0]),
                n_mc=N_MC, seed=seed,
            )
        else:
            tw_ew = run_mc_nd_vec(
                policy_fn=lambda W, g, tau=1.0, _w=ew: _w,   # constant (n,) -> broadcast
                mkt=mkt, n_mc=N_MC, seed=seed,
            )
        all_rows.append({**base, 'method': 'equal_weight',
                         'solve_sec': 0.0, 'train_sec': 0.0,
                         **mc_summary(tw_ew), 'terminal_wealth': tw_ew})

        # ── Neural networks ────────────────────────────────────────────────
        for arch in NN_ARCHS:
            key = (n, arch, seed)
            infer_fn = nn_policies.get(key)
            if infer_fn is None:
                continue
            t_sec = nn_train_sec.get(key, 0.0)
            if n == 1:
                def _nn_1d(w_norm, tau, _fn=infer_fn, _g=GOAL):
                    W = np.asarray(w_norm, float) * _g
                    raw = _fn(W, _g)               # (N_MC,) or scalar
                    return np.asarray(raw, float).ravel()
                tw_nn = run_mc_1d_vec(
                    policy_fn=_nn_1d,
                    mu=float(mkt.mu_ann[0]), r=r, sigma=float(mkt.sigma[0]),
                    n_mc=N_MC, seed=seed,
                )
            else:
                tw_nn = run_mc_nd_vec(
                    policy_fn=lambda W, g, tau=1.0, _fn=infer_fn: _fn(W, g),
                    mkt=mkt, n_mc=N_MC, seed=seed,
                )
            all_rows.append({**base, 'method': arch,
                             'solve_sec': 0.0, 'train_sec': t_sec,
                             **mc_summary(tw_nn), 'terminal_wealth': tw_nn})

    n_rows_n = len([row for row in all_rows if row['n_assets'] == n])
    print(f'  n={n:2d}  MC done  ({n_rows_n} rows)  '
          f'elapsed {time.perf_counter()-t_mc_start:.1f}s')

print(f'\nTotal result rows: {len(all_rows)}')
print(f'Total MC wall time: {time.perf_counter()-t_mc_start:.1f}s')

  n= 1  MC done  (15 rows)  elapsed 1.0s


  n= 5  MC done  (15 rows)  elapsed 2.7s


  n=10  MC done  (15 rows)  elapsed 4.8s


  n=20  MC done  (15 rows)  elapsed 8.2s

Total result rows: 60
Total MC wall time: 8.2s


In [10]:
# ── Cell 10: Aggregate results (mean over seeds) ─────────────────────────
from collections import defaultdict

def agg_rows(rows, metric):
    """Return {method: {n_assets: mean_metric}} averaged over seeds."""
    acc = defaultdict(lambda: defaultdict(list))
    for row in rows:
        if metric in row:
            acc[row['method']][row['n_assets']].append(row[metric])
    return {m: {n: np.mean(v) for n, v in nd.items()} for m, nd in acc.items()}

hit_prob   = agg_rows(all_rows, 'target_hit_prob')
mean_wealth = agg_rows(all_rows, 'mean_terminal_wealth')
shortfall  = agg_rows(all_rows, 'shortfall_mean')

# Build summary table
methods_seen = sorted({r['method'] for r in all_rows})
print('\n=== Mean Target-Hit Probability (averaged over seeds) ===')
header = f"{'Method':<22}" + "".join(f"  n={n}" for n in N_ASSETS_LIST)
print(header)
print('-' * len(header))
for m in methods_seen:
    row_str = f"{m:<22}"
    for n in N_ASSETS_LIST:
        v = hit_prob.get(m, {}).get(n, float('nan'))
        row_str += f"  {v:.3f}"
    print(row_str)


=== Mean Target-Hit Probability (averaged over seeds) ===
Method                  n=1  n=5  n=10  n=20
--------------------------------------------
browne_1d               0.446  nan  nan  nan
browne_nd               nan  0.519  0.415  0.639
equal_weight            0.524  0.575  0.539  0.458
fd_1d                   0.855  nan  nan  nan
fd_nd                   nan  0.835  0.792  0.875
nn_mlp_deep             0.656  0.575  0.711  0.835
nn_mlp_small            0.609  0.576  0.701  0.833


In [11]:
# ── Cell 11: PLOT 1 — Target-hit probability vs n_assets ─────────────────

fig, ax = plt.subplots(figsize=(7, 4.5))
for m in methods_seen:
    meta = METHOD_META.get(m, dict(label=m, color='gray', ls='-', lw=1.5, marker='o'))
    ns = sorted(hit_prob.get(m, {}).keys())
    vals = [hit_prob[m][n] for n in ns]
    if ns:
        ax.plot(ns, vals, color=meta['color'], ls=meta['ls'], lw=meta['lw'],
                marker=meta['marker'], ms=7, label=meta['label'])

ax.axhline(0.5, color='k', ls=':', lw=0.8, alpha=0.5)
ax.set_xlabel('Number of assets n')
ax.set_ylabel('P(W_T ≥ 1.10 × W_0)')
ax.set_title('Target-Hit Probability vs Portfolio Size')
ax.set_xticks(N_ASSETS_LIST)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [12]:
# ── Cell 12: PLOT 2 — Mean terminal wealth vs n_assets ───────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for m in methods_seen:
    meta = METHOD_META.get(m, dict(label=m, color='gray', ls='-', lw=1.5, marker='o'))
    ns = sorted(mean_wealth.get(m, {}).keys())
    vals_w  = [mean_wealth[m][n]  for n in ns]
    vals_sf = [shortfall[m][n]    for n in ns]
    if ns:
        axes[0].plot(ns, vals_w, color=meta['color'], ls=meta['ls'], lw=meta['lw'],
                     marker=meta['marker'], ms=7, label=meta['label'])
        axes[1].plot(ns, vals_sf, color=meta['color'], ls=meta['ls'], lw=meta['lw'],
                     marker=meta['marker'], ms=7)

axes[0].axhline(GOAL, color='k', ls=':', lw=1, label='Goal (1.10)')
axes[0].set_xlabel('n'); axes[0].set_ylabel('E[W_T]')
axes[0].set_title('Mean Terminal Wealth vs n')
axes[0].set_xticks(N_ASSETS_LIST); axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('n'); axes[1].set_ylabel('E[max(1.10 − W_T, 0)]')
axes[1].set_title('Mean Shortfall vs n')
axes[1].set_xticks(N_ASSETS_LIST); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [13]:
# ── Cell 13: PLOT 3 — Terminal wealth histograms for n=1 and n=5 ─────────

fig, axes = plt.subplots(2, len(methods_seen), figsize=(16, 6),
                         sharex=False, sharey=False)
ns_plot = [1, 5]

for row_idx, n in enumerate(ns_plot):
    for col_idx, m in enumerate(methods_seen):
        ax = axes[row_idx, col_idx]
        tw_list = [r['terminal_wealth'] for r in all_rows
                   if r['n_assets'] == n and r['method'] == m]
        if not tw_list:
            ax.axis('off'); continue
        tw = np.concatenate(tw_list)
        meta = METHOD_META.get(m, dict(color='gray', label=m))
        ax.hist(tw, bins=40, color=meta['color'], alpha=0.7, edgecolor='white')
        ax.axvline(GOAL, color='k', lw=1.5, ls='--', label=f'Goal={GOAL}')
        ax.axvline(np.mean(tw), color='red', lw=1.2, ls='-', alpha=0.8)
        hit = (tw >= GOAL).mean()
        ax.set_title(f'{meta["label"]}\nn={n}  P(hit)={hit:.3f}', fontsize=8)
        ax.set_xlabel('W_T' if row_idx == 1 else '')
        ax.set_xlim(0, 2.5)
        ax.grid(True, alpha=0.2)

plt.suptitle('Terminal Wealth Distributions (MC, all seeds pooled)', y=1.02)
plt.tight_layout()
plt.show()

In [14]:
# ── Cell 14: PLOT 4 — Policy π vs wealth for n=1, τ=1 ───────────────────
# Compare FD, Browne, NN (seed=1) at τ=1.0

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
w_norm_plot = np.linspace(0.3, 1.5, 200)
tau_vals = [1.0, 0.5]

for ax, tau in zip(axes, tau_vals):
    # FD
    fd_pol = fd_results[1]['policy']
    pi_fd = np.array([fd_pol(w, tau) for w in w_norm_plot])
    meta = METHOD_META.get('fd_1d', METHOD_META.get('fd_nd', {'color':'steelblue','ls':'-','lw':2,'label':'FD'}))
    ax.plot(w_norm_plot, pi_fd, color=meta['color'], ls=meta['ls'], lw=meta['lw'],
            label=meta['label'])

    # Browne
    br_pol = browne_policies[1]
    pi_br = np.array([br_pol(w, tau) for w in w_norm_plot])
    meta = METHOD_META['browne_1d']
    ax.plot(w_norm_plot, pi_br, color=meta['color'], ls=meta['ls'], lw=meta['lw'],
            label=meta['label'])

    # NNs (seed=1)
    mkt_1 = mkts[1]
    for arch in NN_ARCHS:
        infer_fn = nn_policies.get((1, arch, 1))
        if infer_fn is None:
            continue
        pi_nn = []
        for w in w_norm_plot:
            W = w * GOAL
            pi_nn.append(float(np.asarray(infer_fn(W, GOAL)).ravel()[0]))
        meta = METHOD_META.get(arch, dict(label=arch, color='gray', ls='-', lw=1.5))
        ax.plot(w_norm_plot, pi_nn, color=meta['color'], ls=meta['ls'], lw=meta['lw'],
                label=meta['label'])

    ax.axvline(1.0, color='k', lw=0.8, ls=':', label='Goal (w=1)')
    ax.set_xlabel('w = W / goal')
    ax.set_ylabel('Portfolio weight π')
    ax.set_title(f'Policy comparison — n=1, τ={tau}')
    ax.set_ylim(-0.5, 4)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [15]:
# ── Cell 15: PLOT 5 — Compute cost (solve/train time vs n) ───────────────

fig, ax = plt.subplots(figsize=(7, 4.5))

# FD solve times (now grows with n for fd_nd)
fd_ns   = N_ASSETS_LIST
fd_secs = [fd_results[n]['solve_sec'] for n in fd_ns]
fd_lbls = [fd_results[n]['method_name'] for n in fd_ns]

bars = ax.bar([str(n) for n in fd_ns], fd_secs, alpha=0.75,
              color='steelblue', label='FD HJB solve')
# Annotate bar with method name
for bar, lbl, sec in zip(bars, fd_lbls, fd_secs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            lbl, ha='center', va='bottom', fontsize=7, color='steelblue')

# NN train times
for arch in NN_ARCHS:
    meta = METHOD_META.get(arch, dict(label=arch, color='gray'))
    nn_means = []
    for n in fd_ns:
        times = [nn_train_sec.get((n, arch, s)) for s in SEEDS
                 if nn_train_sec.get((n, arch, s)) is not None]
        nn_means.append(np.mean(times) if times else float('nan'))
    ax.plot([str(n) for n in fd_ns], nn_means,
            color=meta['color'], marker='o', lw=2, label=f"{meta['label']} train")

ax.set_xlabel('Number of assets n')
ax.set_ylabel('Time (seconds)')
ax.set_title('Compute Cost vs Portfolio Size\n(FD now uses proper nD solver for n>1)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


In [16]:
# ── Cell 16: PLOT 6 — Risk vs Performance scatter ────────────────────────
# X axis: mean shortfall (risk), Y axis: target-hit probability (performance)

fig, axes = plt.subplots(1, len(N_ASSETS_LIST), figsize=(14, 4.5), sharey=True)

for ax, n in zip(axes, N_ASSETS_LIST):
    for m in methods_seen:
        meta = METHOD_META.get(m, dict(label=m, color='gray', marker='o'))
        sf_n = shortfall.get(m, {}).get(n)
        hp_n = hit_prob.get(m, {}).get(n)
        if sf_n is not None and hp_n is not None:
            ax.scatter(sf_n, hp_n, color=meta['color'], marker=meta['marker'],
                       s=80, zorder=5, label=meta['label'])
            ax.annotate(meta['label'], (sf_n, hp_n),
                        fontsize=7, xytext=(4, 2), textcoords='offset points')
    ax.set_xlabel('Mean shortfall')
    ax.set_title(f'n={n}')
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Target-hit probability')
fig.suptitle('Risk vs Performance (shortfall vs hit probability)', y=1.02)
plt.tight_layout()
plt.show()

In [17]:
# ── Cell 17: Summary tables ───────────────────────────────────────────────

# Determine which FD method name appears for each n
fd_method_by_n = {n: fd_results[n]['method_name'] for n in N_ASSETS_LIST}
print('FD method per n:', fd_method_by_n)

def _print_table(title, data_dict):
    col_w = 14
    hdr = f"{'Method':<26}" + "".join(f"{('n='+str(n)):>{col_w}}" for n in N_ASSETS_LIST)
    print('\n' + '='*len(hdr))
    print(title)
    print('='*len(hdr))
    print(hdr)
    print('-'*len(hdr))
    for m in sorted(data_dict):
        row = f"{m:<26}"
        for n in N_ASSETS_LIST:
            v = data_dict[m].get(n, float('nan'))
            row += f"{v:>{col_w}.4f}"
        print(row)

_print_table(
    f'Target-Hit Probability  P(W_T >= {GOAL:.2f})  — mean over seeds, N_MC={N_MC}',
    hit_prob)
_print_table('Mean Terminal Wealth  E[W_T]', mean_wealth)
_print_table('Mean Shortfall  E[max(goal - W_T, 0)]', shortfall)

# Compute time table
col_w = 14
hdr = f"{'Method':<26}" + "".join(f"{('n='+str(n)):>{col_w}}" for n in N_ASSETS_LIST)
print('\n' + '='*len(hdr))
print('Compute Time (seconds) — FD solve  /  NN train mean over seeds')
print('='*len(hdr)); print(hdr); print('-'*len(hdr))
row = f"{'FD solve':<26}"
for n in N_ASSETS_LIST:
    row += f"{fd_results[n]['solve_sec']:>{col_w}.2f}"
print(row)
for arch in NN_ARCHS:
    row = f"{arch:<26}"
    for n in N_ASSETS_LIST:
        times = [nn_train_sec.get((n, arch, s)) for s in SEEDS
                 if nn_train_sec.get((n, arch, s)) is not None]
        v = np.mean(times) if times else float('nan')
        row += f"{v:>{col_w}.2f}"
    print(row)


FD method per n: {1: 'fd_1d', 5: 'fd_nd', 10: 'fd_nd', 20: 'fd_nd'}

Target-Hit Probability  P(W_T >= 1.10)  — mean over seeds, N_MC=500
Method                               n=1           n=5          n=10          n=20
----------------------------------------------------------------------------------
browne_1d                         0.4460           nan           nan           nan
browne_nd                            nan        0.5193        0.4153        0.6387
equal_weight                      0.5240        0.5753        0.5387        0.4580
fd_1d                             0.8553           nan           nan           nan
fd_nd                                nan        0.8353        0.7920        0.8747
nn_mlp_deep                       0.6560        0.5753        0.7107        0.8347
nn_mlp_small                      0.6093        0.5760        0.7007        0.8333

Mean Terminal Wealth  E[W_T]
Method                               n=1           n=5          n=10          n=20
---

In [18]:
# ── Cell 18: Save results CSV ─────────────────────────────────────────────
import csv
from pathlib import Path

out_dir = ROOT / 'comparisons' / 'results' / 'full_experiment'
out_dir.mkdir(parents=True, exist_ok=True)

csv_rows = [{k: v for k, v in r.items() if k != 'terminal_wealth'}
            for r in all_rows]

if csv_rows:
    fields = sorted({k for r in csv_rows for k in r})
    path = out_dir / 'notebook03_results.csv'
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        for row in csv_rows:
            w.writerow({k: row.get(k, '') for k in fields})
    print(f'Results saved → {path}')
else:
    print('No results to save.')

Results saved → /Users/ahmedalqubaisi/Desktop/KU/PhD/Code/Claude Code/comparisons/results/full_experiment/notebook03_results.csv


## Interpretation notes

**FD 1D (n=1):** Exact implementation of Dai, Kou, Qian & Wan (2019) — monotone implicit FD scheme with asymptotic viscosity warmstart (condition 18 / Definition A.1 in the paper). The value function V(t,w) and policy π*(w,τ) are provably the unique viscosity solution of the HJB.

**FD nD (n>1):** Proper multi-asset extension. The PDE state is still scalar wealth w (2D PDE in (t,w)); the n-D policy enters through σ²_eff = π*ᵀΩπ* and η_eff = η·π*, which feed the same Thomas tridiagonal. At each grid node: π* = -(V_w/wV_ww) Ω⁻¹η (closed-form QP interior maximiser), then projected onto the leverage-constrained box. Complexity: O(Nw·Nt·n²) vs O(Nw·Nt) for 1D — note the growing solve times in Plot 5.

**FD vs NN (n=1):** Fairest comparison — FD is the theoretical optimum; any NN gap is approximation loss.

**FD vs NN (n>1):** Both now solve the genuine n-asset problem. FD uses the closed-form Merton direction Ω⁻¹η as the unconstrained interior solution; NN learns the policy from simulated trajectories.

**Browne (1D reference):** Analytical solution using equal-risk-weighted 1-D aggregate — a sanity check against the FD 1-D solver; they should closely agree for n=1.

**Leverage:** All methods share the same per-asset box [−5, 3] and aggregate caps (long ≤ 3, short ≤ 5), applied at both train and eval time via `clip_leverage()`.
